# Dataset BMC → Tags MOPEP

1. Carregar dados do BigQuery
2. Tratar e validar tipos
3. Validar duplicações e data leakage
4. Separar train, calibration e test
5. Salvar e recarregar CSV
6. Publicar e recarregar Hugging Face privado

In [ ]:
import ast
import hashlib
import json
import os
import re
from pathlib import Path

import pandas as pd
from google.cloud import bigquery
from sklearn.model_selection import GroupShuffleSplit

> Configs

In [ ]:
PROJECT_ID = "rd-rvn-analytics-prd-01"

SOURCE_TABLE = (
    "rd-rvn-analytics-prd-01."
    "eaas.company_enrichment_latest"
)

RANDOM_STATE = 42
TRAIN_SIZE = 0.70
CALIBRATION_SIZE = 0.15
TEST_SIZE = 0.15
TOP_K = 20

# Pelo menos 1 ocorrência por split.
MIN_SUPPORT_PER_SPLIT = 1


# Evita versionar os dados brutos no Git.
OUTPUT_DIR = Path(
    os.environ.get(
        "MOPEP_BMC_DATA_DIR",
        "/tmp/mopep-bmc-tags",
    )
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID)

# Carregando Dados

In [ ]:
query = f"""
WITH extracted AS (
  SELECT
    LOWER(TRIM(domain)) AS domain,
    enriched_at,
    JSON_VALUE(
      payload,
      '$.mopep.business_model'
    ) AS business_model,
    JSON_VALUE(
      payload,
      '$.mopep.company_tags'
    ) AS tags_raw
  FROM `rd-rvn-analytics-prd-01.eaas.company_enrichment_latest`
),

valid_rows AS (
  SELECT
    domain,
    enriched_at,
    business_model,
    tags_raw
  FROM extracted
  WHERE domain IS NOT NULL
    AND domain != ''
    AND business_model IS NOT NULL
    AND TRIM(business_model) != ''
    AND tags_raw IS NOT NULL
    AND TRIM(tags_raw) NOT IN (
      '',
      '[]',
      'None',
      'nan'
    )
)

SELECT
  domain,
  enriched_at,
  business_model,
  tags_raw
FROM valid_rows
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY domain
  ORDER BY enriched_at DESC
) = 1
"""

raw_df = client.query(query).to_dataframe()

raw_df.shape

In [ ]:
raw_df.assign(
    business_model_preview=(
        raw_df["business_model"]
        .str.replace("\n", " ", regex=False)
        .str.slice(0, 250)
    )
)[
    [
        "domain",
        "enriched_at",
        "business_model_preview",
        "tags_raw",
    ]
].head(10)

In [ ]:
raw_df.dtypes

# Tratamento de Dados

## Funções

In [ ]:
def normalize_domain(value):
    domain = str(value).strip().casefold()

    domain = re.sub(
        r"^https?://",
        "",
        domain,
    )

    domain = re.sub(
        r"^www\.",
        "",
        domain,
    )

    return domain.split("/")[0]


def normalize_business_model(value):
    text = (
        str(value)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    lines = [
        " ".join(line.split())
        for line in text.split("\n")
    ]

    return "\n".join(
        line
        for line in lines
        if line
    ).strip()


def normalize_tag(value):
    return " ".join(
        str(value)
        .strip()
        .casefold()
        .split()
    )


def sha256_text(value):
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

## Tags

In [ ]:
def parse_tags(value):
    if isinstance(value, list):
        parsed = value

    elif value is None or pd.isna(value):
        return [], False

    else:
        text = str(value).strip()
        parsed = None

        # Caso seja JSON válido.
        try:
            candidate = json.loads(text)

            if isinstance(candidate, list):
                parsed = candidate

        except json.JSONDecodeError:
            pass

        # Formato real do EaaS:
        # "['b2b', 'serviço']"
        if parsed is None:
            try:
                candidate = ast.literal_eval(text)

                if isinstance(candidate, list):
                    parsed = candidate

            except (ValueError, SyntaxError):
                pass

        if parsed is None:
            return [], False

    normalized = sorted(
        {
            normalize_tag(tag)
            for tag in parsed
            if tag is not None
            and str(tag).strip()
        }
    )

    return normalized, bool(normalized)

# Tratando Dados

In [ ]:
dataset_df = raw_df.copy()

dataset_df["domain"] = (
    dataset_df["domain"]
    .apply(normalize_domain)
)

dataset_df["enriched_at"] = (
    pd.to_datetime(
        dataset_df["enriched_at"],
        utc=True,
        errors="coerce",
    )
)

dataset_df["business_model"] = (
    dataset_df["business_model"]
    .apply(normalize_business_model)
)

parsed = dataset_df["tags_raw"].apply(
    parse_tags
)

dataset_df["tags"] = parsed.apply(
    lambda result: result[0]
)

dataset_df["parse_ok"] = parsed.apply(
    lambda result: result[1]
)

dataset_df["n_tags"] = (
    dataset_df["tags"].str.len()
)

dataset_df["business_model_chars"] = (
    dataset_df["business_model"].str.len()
)

dataset_df["business_model_hash"] = (
    dataset_df["business_model"]
    .apply(sha256_text)
)

dataset_df["example_id"] = (
    dataset_df.apply(
        lambda row: sha256_text(
            f"{row['domain']}|"
            f"{row['business_model_hash']}"
        ),
        axis=1,
    )
)

In [ ]:
quality_summary = pd.Series(
    {
        "rows": len(dataset_df),
        "unique_domains": (
            dataset_df["domain"].nunique()
        ),
        "parse_failures": (
            (~dataset_df["parse_ok"]).sum()
        ),
        "empty_tags": (
            dataset_df["n_tags"].eq(0).sum()
        ),
        "invalid_dates": (
            dataset_df["enriched_at"]
            .isna()
            .sum()
        ),
        "unique_bmcs": (
            dataset_df[
                "business_model_hash"
            ].nunique()
        ),
    }
)

quality_summary

# Validando Taxonomia

In [ ]:
TAXONOMY_PATH = Path(
    os.environ.get("MOPEP_TAXONOMY_PATH", "/tmp/mopep-tags.csv")
)
taxonomy_df = pd.read_csv(TAXONOMY_PATH)

valid_tags = set(
    taxonomy_df["Tag"]
    .dropna()
    .map(normalize_tag)
)

len(valid_tags)

In [ ]:
dataset_df["n_invalid_tags"] = (
    dataset_df["tags"].apply(
        lambda tags: sum(
            tag not in valid_tags
            for tag in tags
        )
    )
)

# Criando Base Elegível

In [ ]:
eligible_df = dataset_df.loc[
    dataset_df["parse_ok"]
    & dataset_df["domain"].ne("")
    & dataset_df["business_model"].ne("")
    & dataset_df["enriched_at"].notna()
    & dataset_df["n_tags"].gt(0)
    & dataset_df["n_invalid_tags"].eq(0)
].copy()

eligible_df.shape

In [ ]:
eligible_df = (
    eligible_df
    .sort_values("enriched_at")
    .drop_duplicates(
        subset=["domain"],
        keep="last",
    )
    .reset_index(drop=True)
)

eligible_df.shape

## Verificando Conflidos de BMC

In [ ]:
hash_label_sets = (
    eligible_df
    .assign(
        tags_signature=eligible_df["tags"].apply(tuple)
    )
    .groupby("business_model_hash")
    ["tags_signature"]
    .nunique()
)

conflicting_hashes = set(
    hash_label_sets[
        hash_label_sets.gt(1)
    ].index
)

len(conflicting_hashes)

## Analisando Cardinalidade 

In [ ]:
eligible_df["n_tags"].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

## Frequência por tag

In [ ]:
tag_frequency = (
    eligible_df[["domain", "tags"]]
    .explode("tags")
    .groupby("tags", as_index=False)
    .agg(
        rows=("domain", "size"),
        domains=("domain", "nunique"),
    )
    .sort_values(
        ["domains", "tags"],
        ascending=[False, True],
    )
)

tag_frequency

# Gerando Splits

In [ ]:
outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

train_idx, holdout_idx = next(
    outer_split.split(
        eligible_df,
        groups=eligible_df["business_model_hash"],
    )
)

train_df = eligible_df.iloc[train_idx].copy()
holdout_df = eligible_df.iloc[holdout_idx].copy()

In [ ]:
inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=RANDOM_STATE + 1,
)

calibration_idx, test_idx = next(
    inner_split.split(
        holdout_df,
        groups=holdout_df["business_model_hash"],
    )
)

calibration_df = (
    holdout_df.iloc[calibration_idx].copy()
)

test_df = (
    holdout_df.iloc[test_idx].copy()
)

In [ ]:
train_df["split"] = "train"
calibration_df["split"] = "calibration"
test_df["split"] = "test"

## Olhando Tamanhos das Bases geradas

In [ ]:
split_shapes = pd.DataFrame(
    {
        "split": [
            "train",
            "calibration",
            "test",
        ],
        "rows": [
            len(train_df),
            len(calibration_df),
            len(test_df),
        ],
    }
)

split_shapes["percentage"] = (
    split_shapes["rows"]
    / split_shapes["rows"].sum()
)

split_shapes

# Selecionando TOP 20 tags

In [ ]:
top_tags = (
    train_df["tags"]
    .explode()
    .value_counts()
    .head(TOP_K)
    .index
    .tolist()
)

top_tags_set = set(top_tags)

top_tags

## Projetando tags para demais sets

In [ ]:
for split_df in [
    train_df,
    calibration_df,
    test_df,
]:
    split_df["tags_all"] = split_df["tags"]
    split_df["n_tags_all"] = split_df["n_tags"]

    split_df["tags"] = (
        split_df["tags"].apply(
            lambda tags: [
                tag
                for tag in tags
                if tag in top_tags_set
            ]
        )
    )

    split_df["n_tags"] = (
        split_df["tags"].str.len()
    )

In [ ]:
test_df.n_tags_all.min()

# Salvando CSVs

In [ ]:
EXPORT_COLUMNS = [
    "example_id",
    "domain",
    "enriched_at",
    "business_model",
    "business_model_hash",
    "tags",
    "n_tags",
    "business_model_chars",
    "label_source",
    "taxonomy_version",
    "split",
]

for frame in [
    train_df,
    calibration_df,
    test_df,
]:
    frame["label_source"] = (
        "mopep-gpt4o-pseudo-label"
    )
    frame["taxonomy_version"] = "mopep-76-v1"


def prepare_for_csv(frame):
    result = frame[EXPORT_COLUMNS].copy()

    result["tags"] = result["tags"].apply(
        lambda values: json.dumps(
            values,
            ensure_ascii=False,
        )
    )

    result["enriched_at"] = (
        result["enriched_at"]
        .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    )

    return result

split_frames = {
    "train": train_df,
    "calibration": calibration_df,
    "test": test_df,
}

for split_name, frame in split_frames.items():
    output_path = (
        OUTPUT_DIR / f"{split_name}.csv"
    )

    prepare_for_csv(frame).to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    print(output_path)